# Jute-App — App Mode UI (current implementation)

Faithful reproduction of the **as-built** App Mode, grounded in `AppMode.tsx` + `NotebookHeader.tsx` (commit `79d2a0b4`). This is the real, spartan implementation — a counts-only status strip over a vertical stack of *chromeless* frontend-cell outputs — **not** the richer §7c spec mockup (live-dot, cascade-latency, KPI grid), which is not built.

In [ ]:
import math
from IPython.display import HTML, display

# --- server-side baseline so the chart renders even with scripts off (mirrors JS forecast) ---
def forecast(h):
    v, pts = 100.0, []
    for i in range(h):
        v *= 1.045
        pts.append(v * (1 + 0.06 * math.sin(i / 2.2)))
    return pts

W, H, PAD = 620, 170, 14
pts = forecast(12)
lo, hi = min(pts), max(pts)
span = max(1e-9, hi - lo)
sx = W / max(1, len(pts) - 1)
sy = (H - 2 * PAD) / span
X = lambda i: round(i * sx, 1)
Y = lambda i: round(H - PAD - (pts[i] - lo) * sy, 1)
line = "M" + f"{X(0)},{Y(0)}" + "".join(f" L{X(i)},{Y(i)}" for i in range(1, len(pts)))
area = line + f" L{X(len(pts)-1)},{H} L{X(0)},{H} Z"
arr = f"${pts[-1]*0.0345:.1f}M"
growth = f"▲ {((pts[-1]/pts[0]-1)*100):.0f}%"
conf = f"{0.97 - len(pts)*0.006:.2f}"

html = r"""
<!doctype html><html><head><meta charset="utf-8"><style>
  :root{--b:#e5e7eb;--g900:#111827;--g700:#374151;--g600:#4b5563;--g500:#6b7280;--g100:#f3f4f6;--accent:#4f46e5}
  *{box-sizing:border-box}
  body{margin:0;background:#f6f7f9;font:14px/1.5 -apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,system-ui,sans-serif;color:var(--g900)}
  .wrap{max-width:1000px;margin:22px auto;padding:0 16px}
  .win{background:#fff;border:1px solid var(--b);border-radius:10px;overflow:hidden;box-shadow:0 10px 30px rgba(20,30,50,.08)}
  /* ---- title bar (NotebookHeader.tsx) ---- */
  .bar{display:flex;align-items:center;justify-content:space-between;height:46px;padding:0 12px;border-bottom:1px solid var(--b);background:#fff}
  .bl{display:flex;align-items:center;gap:12px;min-width:0}
  .fname{font-size:13px;font-weight:600;color:var(--g900);white-space:nowrap}
  .kstat{display:inline-flex;align-items:center;gap:6px;font-size:12px;color:var(--g600)}
  .dot{width:8px;height:8px;border-radius:50%;background:#10b981;display:inline-block}
  .kstat b{font-weight:600;color:var(--g900)}
  .br{display:flex;align-items:center;gap:2px}
  /* segmented view-mode toggle: rounded border bg-white p-0.5 text-xs */
  .seg{display:flex;border:1px solid var(--b);background:#fff;border-radius:6px;padding:2px;font-size:12px;margin-right:8px}
  .seg button{border:0;background:transparent;border-radius:4px;padding:2px 8px;cursor:pointer;color:var(--g500);font:inherit;transition:background .12s,color .12s}
  .seg button:hover{background:var(--g100);color:var(--g900)}
  .seg button[aria-pressed="true"]{background:var(--g900);color:#fff}
  .ico{width:28px;height:28px;display:grid;place-items:center;border-radius:6px;color:var(--g500);cursor:pointer}
  .ico:hover{background:var(--g100);color:#000}
  .ico svg{width:18px;height:18px}
  /* ---- App mode body (AppMode.tsx): mx-auto max-w-5xl px-8 py-6 ---- */
  .app{max-width:64rem;margin:0 auto;padding:24px 32px}
  .status{display:flex;flex-wrap:wrap;align-items:center;gap:8px;border-bottom:1px solid var(--b);padding-bottom:8px;margin-bottom:16px;font-size:12px;color:var(--g600)}
  .status .app-lbl{font-weight:600;color:var(--g900)}
  .stack{display:flex;flex-direction:column}
  .fcell{border-bottom:1px solid var(--b);padding:16px 0}
  .fcell:last-child{border-bottom:0}
  /* widget outputs (chromeless: no badges / no run controls / no code) */
  .ctrl{display:flex;align-items:center;gap:14px}
  .ctrl label{width:150px;color:var(--g700);font-size:13px}
  .ctrl input[type=range]{flex:1;accent-color:var(--accent)}
  .ctrl .val{width:30px;text-align:right;font-weight:600;font-variant-numeric:tabular-nums}
  .vtitle{font-size:12px;color:var(--g500);margin:0 0 8px}
  .kpis{display:flex;gap:28px;flex-wrap:wrap}
  .kpi .n{font-size:22px;font-weight:700;font-variant-numeric:tabular-nums}
  .kpi .l{font-size:11px;color:var(--g500);text-transform:uppercase;letter-spacing:.04em}
  .up{color:#10b981;font-size:13px;font-weight:600}
  .regions{display:flex;flex-direction:column;gap:6px;font-size:12px;color:var(--g600);min-width:230px}
  .reg{display:flex;align-items:center;gap:8px}
  .reg .lab{width:46px}
  .reg .track{flex:1;height:8px;background:var(--g100);border-radius:4px;overflow:hidden}
  .reg .fill{height:100%;background:#6366f1;border-radius:4px}
  .cap{max-width:1000px;margin:14px auto 30px;font-size:12px;color:var(--g500);line-height:1.6}
  .cap b{color:var(--g700)}
  .cap code{background:#eef0f3;border-radius:3px;padding:1px 5px;font-size:11px}
</style></head><body>
<div class="wrap">
  <div class="win">
    <!-- TITLE BAR -->
    <div class="bar">
      <div class="bl">
        <span class="fname">forecast.ipynb</span>
        <span class="kstat"><span class="dot"></span> python &middot; <b>124 MB</b></span>
      </div>
      <div class="br">
        <div class="seg" role="group" aria-label="Notebook view mode">
          <button title="Notebook cells" aria-pressed="false">Notebook</button>
          <button title="DAG view" aria-pressed="false">DAG</button>
          <button title="App view" aria-pressed="true">App</button>
        </div>
        <span class="ico" title="Settings"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5"><circle cx="12" cy="12" r="3"/><path d="M19.4 15a1.65 1.65 0 0 0 .33 1.82l.06.06a2 2 0 1 1-2.83 2.83l-.06-.06a1.65 1.65 0 0 0-1.82-.33 1.65 1.65 0 0 0-1 1.51V21a2 2 0 0 1-4 0v-.09A1.65 1.65 0 0 0 9 19.4a1.65 1.65 0 0 0-1.82.33l-.06.06a2 2 0 1 1-2.83-2.83l.06-.06a1.65 1.65 0 0 0 .33-1.82 1.65 1.65 0 0 0-1.51-1H3a2 2 0 0 1 0-4h.09A1.65 1.65 0 0 0 4.6 9a1.65 1.65 0 0 0-.33-1.82l-.06-.06a2 2 0 1 1 2.83-2.83l.06.06a1.65 1.65 0 0 0 1.82.33H9a1.65 1.65 0 0 0 1-1.51V3a2 2 0 0 1 4 0v.09a1.65 1.65 0 0 0 1 1.51 1.65 1.65 0 0 0 1.82-.33l.06-.06a2 2 0 1 1 2.83 2.83l-.06.06a1.65 1.65 0 0 0-.33 1.82V9a1.65 1.65 0 0 0 1.51 1H21a2 2 0 0 1 0 4h-.09a1.65 1.65 0 0 0-1.51 1z"/></svg></span>
        <span class="ico" title="Home"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5"><path d="M3 9.5 12 3l9 6.5"/><path d="M5 10v10h14V10"/></svg></span>
        <span class="ico" title="New"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5"><path d="M12 5v14M5 12h14"/></svg></span>
      </div>
    </div>

    <!-- APP MODE BODY -->
    <div class="app">
      <div class="status" role="status" aria-label="App mode status">
        <span class="app-lbl">App</span>
        <span>3 frontend cells</span>
        <span>1 running</span>
      </div>
      <div class="stack">
        <!-- frontend control cell -> chromeless CellOutput -->
        <section class="fcell">
          <div class="ctrl">
            <label for="hz">Horizon (months)</label>
            <input id="hz" type="range" min="1" max="24" value="12">
            <span class="val" id="hzv">12</span>
          </div>
        </section>
        <!-- frontend view cell (chart) -->
        <section class="fcell">
          <p class="vtitle">Revenue projection</p>
          <svg id="chart" viewBox="0 0 __W__ __H__" width="100%" height="180" preserveAspectRatio="none">
            <defs><linearGradient id="fade" x1="0" x2="0" y1="0" y2="1">
              <stop offset="0" stop-color="#6366f1" stop-opacity=".22"/><stop offset="1" stop-color="#6366f1" stop-opacity="0"/>
            </linearGradient></defs>
            <path id="area" d="__AREA__" fill="url(#fade)"/>
            <path id="line" d="__LINE__" fill="none" stroke="#4f46e5" stroke-width="2"/>
          </svg>
        </section>
        <!-- frontend view cell (KPIs + regions) -->
        <section class="fcell">
          <div style="display:flex;gap:40px;flex-wrap:wrap;align-items:flex-start">
            <div class="kpis">
              <div class="kpi"><div class="n" id="arr">__ARR__</div><div class="l">ARR &nbsp;<span class="up" id="growth">__GROWTH__</span></div></div>
              <div class="kpi"><div class="n" id="conf">__CONF__</div><div class="l">Confidence</div></div>
            </div>
            <div class="regions">
              <div class="reg"><span class="lab">APAC</span><span class="track"><span class="fill" style="width:82%"></span></span></div>
              <div class="reg"><span class="lab">EU</span><span class="track"><span class="fill" style="width:54%"></span></span></div>
              <div class="reg"><span class="lab">US</span><span class="track"><span class="fill" style="width:38%"></span></span></div>
            </div>
          </div>
        </section>
      </div>
    </div>
  </div>
  <p class="cap">
    <b>This is the as-built App Mode</b> (<code>AppMode.tsx</code> + <code>NotebookHeader.tsx</code>, commit <code>79d2a0b4</code>):
    a centered <code>max-w-5xl</code> column with a <b>counts-only status strip</b> (<code>App &middot; N frontend cells &middot; running/failed/stale</code>)
    over a <b>vertical stack of frontend cells in document order</b>, each rendered <b>chromeless</b> (output only — no code, run buttons, or DAG badges).
    The <code>Notebook / DAG / App</code> segmented toggle lives in the title bar; switching to <code>App</code> hides everything but frontend-cell outputs.<br>
    <b>Not built</b> (spec §7c mockup): the &ldquo;● live&rdquo; indicator, last-cascade-latency readout, and the KPI/grid layout engine.
    Layout is a plain stack — no grid/<code>props.layout</code> yet. The slider here is a static client-side stand-in; the real control pushes a port and cascades through the in-process engine.
  </p>
</div>
<script>
(function(){
  var W=__W__,H=__H__,PAD=__PAD__;
  function forecast(h){var v=100,p=[];for(var i=0;i<h;i++){v*=1.045;p.push(v*(1+0.06*Math.sin(i/2.2)));}return p;}
  function paths(p){var lo=Math.min.apply(null,p),hi=Math.max.apply(null,p),sp=Math.max(1e-9,hi-lo);
    var sx=W/Math.max(1,p.length-1),sy=(H-2*PAD)/sp;var x=function(i){return (i*sx).toFixed(1);},y=function(i){return (H-PAD-(p[i]-lo)*sy).toFixed(1);};
    var l="M"+x(0)+","+y(0);for(var i=1;i<p.length;i++)l+=" L"+x(i)+","+y(i);
    return {line:l,area:l+" L"+x(p.length-1)+","+H+" L"+x(0)+","+H+" Z",last:p[p.length-1],first:p[0],n:p.length};}
  var hz=document.getElementById('hz');
  function draw(){var h=+hz.value,p=forecast(h),q=paths(p);
    document.getElementById('hzv').textContent=h;
    document.getElementById('line').setAttribute('d',q.line);
    document.getElementById('area').setAttribute('d',q.area);
    document.getElementById('arr').textContent="$"+(q.last*0.0345).toFixed(1)+"M";
    document.getElementById('growth').textContent="▲ "+((q.last/q.first-1)*100).toFixed(0)+"%";
    document.getElementById('conf').textContent=(0.97-q.n*0.006).toFixed(2);
  }
  if(hz) hz.addEventListener('input',draw);
})();
</script>
</body></html>
"""

html = (html.replace("__W__", str(W)).replace("__H__", str(H)).replace("__PAD__", str(PAD))
            .replace("__AREA__", area).replace("__LINE__", line)
            .replace("__ARR__", arr).replace("__GROWTH__", growth).replace("__CONF__", conf))
display(HTML(html))